# CTC Model Training Pipeline
This notebook implements the Connectionist Temporal Classification (CTC) pipeline. 
Unlike the sliding window approach, this trains the `CTCModel` sequentially on entire audio recordings using PyTorch's native `CTCLoss`.

In [1]:
import sys

assert sys.version_info >= (3, 10)
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    !git clone https://github.com/stachuapa123/ASR_project.git
    %cd ASR_project
    # !git checkout <YOUR_BRANCH_NAME>  # Uncomment and set this to your branch if needed
    !pip install -q torchmetrics
    from google.colab import drive

    drive.mount("/content/drive")

    # Extract data securely if on Colab
    !mkdir -p "/content/asr_data"
    !unzip -q "/content/drive/MyDrive/asr_data.zip" -d "/content/asr_data"
    DATA_DIR = "/content/asr_data"
else:
    # Local path
    %load_ext autoreload
    %autoreload 2
    DATA_DIR = "../data"  # Update to your local subset or AutorskieDane

In [2]:
import torch
from torch.utils.data import DataLoader

from src.ctc.config import CTCConfig as C
from src.ctc.model import CTCModel
from src.ctc.dataset import CTCDataset, waveform_collate_fn
from src.ctc.features import CTCFeatureExtractor
from src.ctc.augmentation import SpecAugment
from src.ctc.training import train_ctc, EarlyStopping

In [3]:
# Hyperparameters
PCT_VAL = 0.15
BATCH_SIZE = 16
N_EPOCHS = 100
LR = 1e-3
MAX_LR = 1e-3
WEIGHT_DECAY = 1e-4
PCT_START = 0.2
NUM_WORKERS = 4

device = C.get_device()
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Waveform mode: the dataset returns raw waveforms and feature extraction
# (log-mel + augmentation) runs batched on the GPU via CTCFeatureExtractor.
dataset = CTCDataset(
    data_root=DATA_DIR,
    apply_augmentations=True,
    return_waveform=True,
)

n_total = len(dataset)
n_val = max(1, int(PCT_VAL * n_total))
n_train = n_total - n_val
generator = torch.Generator().manual_seed(42)
train_set, val_set = torch.utils.data.random_split(
    dataset,
    [n_train, n_val],
    generator=generator,
)
print(f"Train items: {len(train_set)} | Val items: {len(val_set)}")

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=waveform_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=waveform_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

Train items: 8754 | Val items: 1544


In [5]:
model = CTCModel()
objective = torch.nn.CTCLoss(blank=C.BLANK_IDX, zero_infinity=True)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    steps_per_epoch=len(train_loader),
    epochs=N_EPOCHS,
    pct_start=PCT_START,
)
scaler = torch.amp.GradScaler(
    device=device.type,
    enabled=(device.type == "cuda"),
)
es = EarlyStopping(patience=10)
feature_extractor = CTCFeatureExtractor(spec_augment=SpecAugment()).to(device)

In [6]:
model = train_ctc(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    objective=objective,
    device=device,
    n_epochs=N_EPOCHS,
    feature_extractor=feature_extractor,
    scheduler=scheduler,
    scaler=scaler,
    early_stopping=es,
    save_best_to="../trained_models/ctc_test.pt",
    use_amp=(device.type == "cuda"),
    step_scheduler_per_batch=True,
)

Epoch   1/100 | Train Loss: 3.8900 | Val Loss: 3.3673 | Val PER: 1.0000 | LR: 4.6e-05 [BEST]          
Epoch   2/100 | Train Loss: 3.1135 | Val Loss: 2.6850 | Val PER: 0.9937 | LR: 6.3e-05 [BEST]          
Epoch   3/100 | Train Loss: 2.3951 | Val Loss: 2.0163 | Val PER: 0.5667 | LR: 9.2e-05 [BEST]          
Epoch   4/100 | Train Loss: 1.8739 | Val Loss: 1.5686 | Val PER: 0.4287 | LR: 1.3e-04 [BEST]          
Epoch   5/100 | Train Loss: 1.5804 | Val Loss: 1.3129 | Val PER: 0.3660 | LR: 1.8e-04 [BEST]          
Epoch   6/100 | Train Loss: 1.4071 | Val Loss: 1.2006 | Val PER: 0.3406 | LR: 2.4e-04 [BEST]          
Epoch   7/100 | Train Loss: 1.2666 | Val Loss: 1.1290 | Val PER: 0.3301 | LR: 3.0e-04 [BEST]          
Epoch   8/100 | Train Loss: 1.1603 | Val Loss: 0.9748 | Val PER: 0.2754 | LR: 3.7e-04 [BEST]          
Epoch   9/100 | Train Loss: 1.0611 | Val Loss: 0.9715 | Val PER: 0.2704 | LR: 4.4e-04 [BEST]          
Epoch  10/100 | Train Loss: 0.9835 | Val Loss: 0.8671 | Val PER: 0.2488 |

KeyboardInterrupt: 